# Decode attention over a paged KV cache

A paged KV cache stores keys and values in a shared pool of fixed-size **pages**. Each sequence owns a list of page ids (its **block table**), so memory is allocated one page at a time instead of reserving `max_len` slots per request. Attention must gather a sequence's pages back into order.

Write `paged_decode(q, k_pages, v_pages, block_table, lengths)`:

- `q`: `(B, H, d)`, one decode query per sequence.
- `k_pages`, `v_pages`: `(P, page_size, H, d)`, the shared pool.
- `block_table`: `(B, n_blocks)` int. Logical position $s$ of sequence $b$ lives in page `block_table[b, s // page_size]` at slot `s % page_size`. Entries for blocks a sequence does not use are `-1`.
- `lengths`: `(B,)` int, `1 <= lengths[b] <= n_blocks * page_size`.

Return `(out, k_seq)`:

- `k_seq`: `(B, n_blocks * page_size, H, d)`, each sequence's keys in logical order, with positions `>= lengths[b]` set to 0.
- `out`: `(B, H, d)`, softmax attention $\mathrm{softmax}(q k^\top / \sqrt d)\, v$ per head over positions `< lengths[b]`.

float32. Do not rely on what indexing with `-1` does: unused entries must not affect the result.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def paged_decode(q, k_pages, v_pages, block_table, lengths):
    """Return (out (B, H, d), k_seq (B, n_blocks * page_size, H, d))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[0.0]]], "dtype": "float64"}, {"__nd__": [[[[0.0]], [[1.0]]], [[[2.0]], [[3.0]]], [[[4.0]], [[5.0]]], [[[6.0]], [[7.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0]], [[1.0]]], [[[2.0]], [[3.0]]], [[[4.0]], [[5.0]]], [[[6.0]], [[7.0]]]], "dtype": "float64"}, {"__nd__": [[2, 0]], "dtype": "int64"}, {"__nd__": [3], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[3.0]]], "dtype": "float32"}, {"__nd__": [[[[4.0]], [[5.0]], [[0.0]], [[0.0]]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[0.0]], [[0.0]]], "dtype": "float64"}, {"__nd__": [[[[0.0]], [[1.0]]], [[[2.0]], [[3.0]]], [[[4.0]], [[5.0]]], [[[6.0]], [[7.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0]], [[1.0]]], [[[2.0]], [[3.0]]], [[[4.0]], [[5.0]]], [[[6.0]], [[7.0]]]], "dtype": "float64"}, {"__nd__": [[3, -1], [1, 2]], "dtype": "int64"}, {"__nd__": [2, 4], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[6.5]], [[3.5]]], "dtype": "float32"}, {"__nd__": [[[[6.0]], [[7.0]], [[0.0]], [[0.0]]], [[[2.0]], [[3.0]], [[4.0]], [[5.0]]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[1.5746411431681302, 0.7432833867028191, -0.9698888175576575, -0.2120044493626771], [-0.2867299190012126, 2.3624189202137407, -0.943225489608639, 1.3757948597156504]], [[0.12317992062651277, 1.0218538102952426, -0.0027781216332353766, 0.3927701046524877], [0.491156064712792, 0.10953869409072106, 0.6512592984651386, 0.3496303616197618]]], "dtype": "float64"}, {"__nd__": [[[[1.2328190417642915, 1.8074424397926976, -0.08832984255347656, -0.1872296676125356], [0.4106914898142733, -1.6574989126064583, 0.3081900463490193, -0.524235693173715]], [[-0.822935739443747, -1.5787501417612935, -0.22460757394585165, 0.5098854894376852], [-0.5283039225093467, -0.9098222824092372, -0.5625451549455628, -0.15138536455592896]], [[-1.1588554465764798, -0.534120971120513, -1.0754254370007643, -0.12611617018133786], [1.7445967926863417, 0.1654229334653844, 1.991804368947761, -0.6931362778325035]]], [[[-1.1360708805900552, -0.2839527224539722, 0.08517447239325363, 0.37412241910263816], [0.3865577313799712, -0.31601773316141096, 0.8858351496326213, -0.08926181613751151]], [[0.7000924289292733, 0.4486394595197229, 0.25171390091305645, 0.22110198826650268], [-0.7227628693585002, 0.5410272721579086, 0.58090355163021, -0.48162852108049914]], [[2.5429431556159434, 0.16510561371719804, -1.284045224638924, -1.5712734851726575], [-0.8493941091136205, -2.1907090298801184, 0.40537266623124274, -1.063768519850836]]], [[[0.3875831316452551, -1.375199133922121, 0.22340479558120588, 1.0767381896129868], [0.5853709423785727, -1.7404030543426012, 0.061422942801712954, -0.37383440463813067]], [[1.1763619171014466, 0.41875415528092563, 1.892800751444767, 0.2087660783314162], [-1.0445133359135457, -0.8646300000215755, 1.2351174399762312, -0.9437731203231157]], [[0.19449691788139745, 2.1841928188365816, 0.03790757874869285, -0.8415980855733193], [0.9109362742821808, 0.42713739807442513, -1.1460636494810137, 0.8151963882050687]]], [[[0.23675661418432425, 0.7506349387877089, -0.6045710468116573, 1.1820312695503685], [-0.46228239849538183, 1.2869072202893832, -1.0665470418784737, 0.346202452961344]], [[0.3829970229390242, -1.247813798771936, -1.867266823717423, -0.6642093078004342], [-0.7584866925124449, -0.6192536054646688, -0.11660305960203166, 2.270774559169564]], [[0.22496601106091482, 1.6592202995840934, 0.40840723250640504, 1.156279882691092], [0.6212008743370198, 0.11430211710038234, 0.5362943612103488, 2.2310154762988987]]], [[[-1.5882795191331023, 0.3000693526320025, 0.6782360773858063, -1.1667557463677165], [0.5897069565477524, 0.23274009796536124, 1.2867345719764205, 0.6709501944315489]], [[0.026803546257372364, 0.32604867555125033, -2.1284897470268875, 2.0513719405307955], [0.2742931600921251, -1.3648635929117345, -0.9604207054805258, -0.20716318436801218]], [[-0.5708462321239673, 0.17913066011662224, -0.6813218411683131, 0.5118444238332029], [-0.47533370819623755, -0.09638538591639903, -1.4076154500791598, -0.7797791037556863]]], [[[-0.1655311039192391, -0.9806907089947733, -0.2935412684991226, -0.37276845030375894], [-1.6310229041056268, 1.0146115415722108, -1.4351033390128491, 1.62658623536235]], [[0.9455815676517173, 0.4927778126078554, -0.10421981995747179, -0.9216464076127447], [1.573235436397624, 1.9639528935049797, -0.7431953364154976, 1.7710644980052206]], [[1.1074839092425983, 0.8629993223288605, 0.5011623760697407, -0.27950710982942895], [-0.08740725891600336, 0.2283092409563164, 1.3292675107693794, -1.2734960110897269]]]], "dtype": "float64"}, {"__nd__": [[[[0.22025764296612885, -1.0757347193451983, 0.08664031909029198, -0.2648191112197345], [-0.734323443040384, 0.6127082450267287, 0.6818739987129918, -1.2505224143068572]], [[-0.7303632989740125, 0.47155761497328974, -0.5059307853616831, 0.29875505696504817], [-2.78261971124263, -0.4015138222182967, -0.33886765601456276, 0.2688905491141114]], [[2.1990198573712747, -0.26980238864101574, -0.1740030556087151, 1.4162563331644122], [-0.13224739222742846, 1.6360338005770318, 0.7491397927023967, -1.7833568082169045]]], [[[0.1848027413936195, 1.5819143716929263, 0.22048884513058445, 0.2950188428567004], [0.175351595461967, 0.14359026641505127, 1.2519022677568883, 0.5485928463698302]], [[0.24983650658496312, 2.3467199945549906, -1.455890983383913, -0.6514802147644286], [-0.3052127818706818, -0.372317170721315, -1.7103046452699644, 2.6656388991067135]], [[-0.08838287589258985, 0.5248323680550954, -0.01615477447781413, -1.193060323082117], [2.3118615278795804, -0.12143144612001598, 0.13231086678498935, -0.043901029079651634]]], [[[0.2998382277069786, 1.4088855277897172, -0.04684042884785556, -3.370786830400435], [-0.6645903907486291, -1.0760925182764627, 0.559657930308173, -0.41123111391217915]], [[0.4847614178726964, 0.17568167985488428, -0.5294433848477941, 0.750996857754231], [-0.2990340947561143, -1.193223213371992, -1.534643444265342, -1.013283213559938]], [[-2.6303480409201474, -0.07250758736465224, 0.19504132270977223, -1.2283801062710602], [-2.3350521818847425, -1.1698197999059288, 0.8864253094492558, -0.2194136229490016]]], [[[0.04912982955201279, 0.20002382918892003, -0.04704913644411588, -0.21980215768708072], [0.5960519988994732, 0.5672640371175585, -0.7775973248355932, -0.7487765721282506]], [[-0.13030707943474681, 0.2925779270600447, 0.5388064138224908, -0.13296943347982076], [-0.1938213127425998, -1.1859234288475822, 1.1663472106524877, -0.002190165835947568]], [[0.7458892037431221, -0.3133989243602814, -1.0886853797464664, -0.5701161341695636], [-0.5965029541106073, 0.48886836013900636, -0.38722725548067627, 1.0765336282316957]]], [[[0.5370689808459439, 2.5102881315030854, -1.6790241276647815, 1.3980858393170656], [-0.5315278295717231, -0.030994868634583388, 0.29774659696669636, 1.1241962919657005]], [[0.09163875982087606, -0.4220482933696502, 1.7143782043922384, -0.6684752643267876], [0.9928667758108061, -0.1743190251710952, 1.2415355258045806, -0.022522035866328758]], [[-0.8133484151273919, -1.1185353160430629, -0.6792063335066112, 0.7187519281811856], [1.423952816796723, 0.0932042495611786, 0.34838415251719035, -1.4713251004941599]]], [[[-0.17167158203890956, 1.0984113519853334, 1.0533074112612788, -1.2364585786002502], [-0.43609074745251836, -0.5551649635790397, 0.29989226951023895, -0.6946499851718569]], [[0.018031232159339057, 1.3320165936932338, 0.1733082607520546, -0.353932226692057], [0.15745955741086853, -1.4358361000599833, -0.07558219739707056, 0.1130614442431498]], [[1.6480010083069785, -0.9752577047034277, 1.0702750047679679, 0.5756720539073316], [1.8006878985542747, 1.5503221728984373, 0.5141887916390588, -0.19599212538929092]]]], "dtype": "float64"}, {"__nd__": [[5, 0, 2], [1, 4, -1]], "dtype": "int64"}, {"__nd__": [7, 6], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[0.5044571161270142, -0.2168123424053192, 0.32369354367256165, -0.23781700432300568], [-0.08701284974813461, -1.061633586883545, 0.07501445710659027, -0.20824790000915527]], [[0.011699999682605267, 0.7369850277900696, -0.1409417688846588, -0.13290433585643768], [0.2976655066013336, -0.05987835302948952, 0.29833394289016724, 0.7709396481513977]]], "dtype": "float32"}, {"__nd__": [[[[-0.16553109884262085, -0.9806907176971436, -0.29354128241539, -0.37276846170425415], [-1.6310229301452637, 1.0146114826202393, -1.4351032972335815, 1.6265861988067627]], [[0.9455815553665161, 0.49277782440185547, -0.10421981662511826, -0.9216464161872864], [1.5732353925704956, 1.963952898979187, -0.7431953549385071, 1.7710645198822021]], [[1.1074838638305664, 0.8629993200302124, 0.5011623501777649, -0.2795071005821228], [-0.08740726113319397, 0.2283092439174652, 1.3292675018310547, -1.2734960317611694]], [[1.2328190803527832, 1.8074424266815186, -0.08832984417676926, -0.18722966313362122], [0.410691499710083, -1.6574989557266235, 0.3081900477409363, -0.5242356657981873]], [[-0.8229357600212097, -1.5787501335144043, -0.22460757195949554, 0.5098854899406433], [-0.5283039212226868, -0.9098222851753235, -0.5625451803207397, -0.1513853669166565]], [[-1.1588554382324219, -0.5341209769248962, -1.075425386428833, -0.12611617147922516], [1.7445968389511108, 0.1654229313135147, 1.9918043613433838, -0.6931362748146057]], [[0.3875831365585327, -1.3751990795135498, 0.22340479493141174, 1.0767382383346558], [0.5853709578514099, -1.7404030561447144, 0.06142294406890869, -0.37383440136909485]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]], [[[-1.1360708475112915, -0.2839527130126953, 0.08517447113990784, 0.37412241101264954], [0.3865577280521393, -0.316017746925354, 0.8858351707458496, -0.08926181495189667]], [[0.7000924348831177, 0.448639452457428, 0.25171390175819397, 0.22110198438167572], [-0.7227628827095032, 0.5410272479057312, 0.5809035301208496, -0.48162850737571716]], [[2.542943239212036, 0.16510561108589172, -1.2840452194213867, -1.5712734460830688], [-0.8493940830230713, -2.190709114074707, 0.405372679233551, -1.0637685060501099]], [[-1.5882794857025146, 0.3000693619251251, 0.6782360672950745, -1.1667557954788208], [0.5897069573402405, 0.2327401041984558, 1.2867345809936523, 0.670950174331665]], [[0.026803545653820038, 0.32604867219924927, -2.1284897327423096, 2.0513720512390137], [0.2742931544780731, -1.364863634109497, -0.9604207277297974, -0.20716318488121033]], [[-0.5708462595939636, 0.1791306585073471, -0.6813218593597412, 0.5118443965911865], [-0.47533372044563293, -0.09638538956642151, -1.4076154232025146, -0.7797790765762329]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[1.62131080323827, 1.8799812131838187, -0.12828235600687177, 1.8197655966867985, 0.22257009812133635, 1.027306575955318, -0.900006553532683, 0.6902116701689707]], [[0.14792891799713967, 0.06862574408621153, -0.5628077127558095, -0.4426090995608147, 0.6564068016526381, -0.2129022941535824, -1.74886094729239, -0.9900623026319665]], [[1.3921013232767878, 0.09178125299164155, -0.1176947088400653, 1.6374462784229382, -1.805148448606515, -1.0320862110371847, 0.7014484129536175, 1.1732809327780787]]], "dtype": "float64"}, {"__nd__": [[[[0.3479893906010453, -1.1470914179803955, 1.0219467775588984, 0.23404602290197407, -0.8708233584216591, -2.1425738231366918, 0.019310842646497565, -0.7578818593236448]], [[-0.963149167446121, -0.8164395658324995, -0.403114621193118, -0.5547810531522746, 0.09327373212282386, 1.3676627244812698, -2.2305142238617606, -0.9793414349840619]], [[0.009236526074225343, 0.03132611522376319, 0.5042862577782419, -0.24340831475556998, -0.2129805155411493, 0.3005010963170778, 1.0175664778232425, 0.6195608857550361]], [[-0.3941349280645121, 0.736650010004568, 0.4039818939880654, -0.7567050595355779, 0.8514222045954762, 0.759430358648976, -0.9627148708055201, 0.9945997931369627]]], [[[-0.2573989257528388, 1.991979154422499, -0.9957233947341623, -0.03232672559708717, -0.1812969658730386, -1.0313743659779548, -0.5409165568607083, 0.18286783623163103]], [[-0.38830555811313455, 0.24725558878532097, -0.5627174778023206, 0.32882596639171346, 0.24212321776980494, -1.0375146022438104, -2.7078871789390337, 0.36336381031580306]], [[-1.0580006103422213, -4.685291268202864, 0.928638230072882, -0.3298652413352513, 0.6736017986810107, 0.8814099890513736, 3.204355130585023, 0.38539957426678306]], [[1.001315434540215, 1.4020072509636368, 1.8176027876259166, -0.03851103269428163, 0.23633622407440058, 0.8209565922084492, -1.3858610065490449, 0.6251533293498069]]], [[[1.5713118746606944, 0.18823588176671366, 1.0011441747892782, -0.8182663137884161, 0.9296925355170065, -1.621872816552439, -0.3184634048926212, 1.1756529944865217]], [[-0.7054431504454319, 0.5931043189423127, -1.067927933551086, 0.08812177541922656, 0.12540509184844828, -0.30736380650135364, -0.874426205752552, -0.3944861746879388]], [[-1.0484990110143502, -0.8986002444572808, -1.1938445392571566, -0.3069350987931802, 1.1766713614164783, 2.4537768226739898, 0.5493519790114207, 0.4376281963512965]], [[-0.8518537172308596, -0.7007747582736603, 0.8302022438224563, 1.1690584653299025, -0.8054897626951746, -0.43780743187388993, 1.2180014187357713, 1.2610498979913367]]], [[[0.9670621633031591, -0.4502611692258008, -0.45757814598214847, 0.059052398541128426, -2.4073487769406117, 0.32037190402308113, 0.9106119449273173, 0.10448248391583169]], [[0.31660969538452843, -1.042112881613743, 0.37535426351182266, 0.3311896775658899, -0.5744143554869527, 0.15209285794160543, -1.2250162104619469, -0.7944929142267736]], [[-0.04402835758375899, -0.8324891514221003, -1.0727643274427656, -1.1896279807145307, 0.34108578117514743, 1.026548884151484, 1.2667452067015939, 0.008510382066666237]], [[-0.20542397638950943, -0.016255156981131447, 0.9043656809532198, -1.0108083451821772, -0.05690971015204637, 0.12237723934255906, -0.3618774541037053, 1.2302177289007197]]], [[[0.17199710027773982, -0.47490925134216155, 1.4037201671761854, -1.2295411779149215, -0.40415230479217007, -0.302105586311098, -0.8899335253844772, 1.0011768026547818]], [[-1.3202907013128486, -0.047616816306246826, -1.1480863098748442, -0.5248043345160446, 0.994309772888166, -1.1493183068063217, -1.0245747175632092, 0.5307970751114715]], [[-0.6790192349657492, 0.4735883449704902, 1.6077430023086634, 0.04546275137266199, 0.38747132778564514, 0.5995754594227106, 0.36248454203437425, -3.059925411441666]], [[-1.100758938244725, 1.8496284340682168, 0.40399864956008086, -0.711199093596586, 0.29445590894608653, 0.6573604288355155, -0.6256714950139831, 0.6714551435166187]]]], "dtype": "float64"}, {"__nd__": [[[[1.7585246396803251, 0.6849174015688669, 0.7237349788146868, 0.18081819683088313, 0.7987837725313719, -0.6562454526907721, 1.499769981702536, 0.4924756141243058]], [[1.3182418368983444, 1.9715458194084818, -0.9399808580490141, 0.04648352022089345, -1.2161218850868505, 2.1880102403240116, -0.49111231919139686, -0.9380386550287374]], [[0.5731238529858683, 0.15396924552882382, 1.0894983337272, -0.8019920986730715, 1.1759603123651954, 1.2165236535746333, -1.6232195902423436, -0.33834827962316694]], [[-1.0981646336989943, -1.0996534040431856, -1.102197118653191, -0.6694407972813652, -0.6057620626013438, -0.8834204286770616, 2.3167402661450693, -1.443313736547179]]], [[[-0.5270793429196685, -0.6175715977321008, -2.5211649795646114, 0.49000017589286726, 0.575730789133456, 0.11314809547807715, -1.495446301204332, 0.042399238872063476]], [[-1.4510069520706483, -0.2308270982075071, -0.051547945387897746, 0.22922126833863146, -0.28553722002896265, 0.3022040437088754, 0.39606423564970294, -0.08502196854320478]], [[-0.7729557317500808, 0.5763427038729221, 0.6953394372852897, 1.0310945058919956, -0.0018739527452244253, 0.9519726162046341, -0.6968606863809321, -0.12200838667952692]], [[0.19629043925022793, 1.7178864705528925, -0.4842671972989275, -0.1458709183663478, -0.5915634773488055, 1.1075798290944885, 0.7560624457791167, -0.1518237378267948]]], [[[-1.4847168259439696, 0.747109444315093, -1.2329621243869655, 0.8176746792557876, -1.5597639418539666, 0.7186068613524211, -0.63842626486969, -1.4757510705582273]], [[0.48601263995954835, 0.06779833738064875, -2.2572241634241443, 1.3086263887662302, -0.28583809548748323, -1.7325466229837077, 0.0493992794274074, 0.529634765184706]], [[0.19859607687725506, -0.6010475741307065, 0.7623470490670095, -0.5837086953686035, -1.1296082789406154, 1.0222716048369482, -0.5172338900759362, -0.4897787803403812]], [[1.4618328264450626, -1.039917015552758, 1.484441597518495, -0.1950856124373444, 0.17038146518882097, 0.023469097667393312, -0.5501012659568211, 0.3345316943621903]]], [[[1.1967068213200973, -1.0535194995436317, -0.9721101594359809, 0.1856836153447373, -0.438670905753179, 1.869443751285454, 0.4004940288487152, -0.1836256210910374]], [[-0.9588917020483002, 0.13796926824786204, -0.6715716396379724, 0.9536170374812901, -1.4246119551676801, 0.021436980816945755, 0.6619178737469374, -1.1401039939380395]], [[-0.9484437807685275, -0.961897184429497, 0.7089052468899443, 0.27201361018338893, -1.864792745174152, 0.0805168195754283, 0.17333779515947317, 0.5034921552033106]], [[1.2965578422912567, -1.680531592184256, 0.9813708611972458, -1.4507269973270929, -0.7180643126655215, -0.6140990963544882, -0.3546172739863653, -0.43018800219767184]]], [[[-0.1316062441531609, 0.6014863173678617, 0.9443220145876112, 0.6187879905065031, -0.15046836396330215, -0.7468744083450605, 0.8962089920159839, 0.6411084553907437]], [[0.5491887291327594, 0.22173740323010677, 0.6886050053212818, -1.9794911242325857, 0.869360501987822, 0.5565310839391474, -0.6418472503806664, 0.9114777423320349]], [[-0.6047790731269551, -0.8787641974160343, 0.24989675379031895, 1.3361052853513693, 0.31395242463656875, -1.560024935045845, -0.6782172501651925, 0.6182621929242446]], [[0.4884457982953561, -0.6947703600497817, 0.3223585000845514, 1.4616721563063557, -0.5144897389270865, 1.0517416378982687, 0.23924229690422114, -0.40742759240873494]]]], "dtype": "float64"}, {"__nd__": [[4, -1], [0, 3], [2, 1]], "dtype": "int64"}, {"__nd__": [1, 8, 5], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[-0.13160625100135803, 0.601486325263977, 0.9443219900131226, 0.6187880039215088, -0.1504683643579483, -0.7468743920326233, 0.8962090015411377, 0.6411084532737732]], [[0.5189073085784912, 0.6614078283309937, -0.5182818174362183, 0.022042661905288696, -0.9312978982925415, 0.8977819085121155, 0.23843549191951752, -0.776418149471283]], [[0.479718953371048, -0.5399201512336731, 0.08402866870164871, 0.19917859137058258, -0.16434642672538757, 0.09071359783411026, -0.6787360906600952, -0.07963094860315323]]], "dtype": "float32"}, {"__nd__": [[[[0.1719971001148224, -0.474909245967865, 1.4037201404571533, -1.2295411825180054, -0.4041523039340973, -0.302105575799942, -0.8899335265159607, 1.0011768341064453]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], [[[0.34798938035964966, -1.1470913887023926, 1.0219467878341675, 0.23404602706432343, -0.8708233833312988, -2.142573833465576, 0.0193108431994915, -0.7578818798065186]], [[-0.9631491899490356, -0.8164395689964294, -0.4031146168708801, -0.5547810792922974, 0.09327372908592224, 1.3676626682281494, -2.2305142879486084, -0.979341447353363]], [[0.009236525744199753, 0.031326115131378174, 0.5042862296104431, -0.24340830743312836, -0.2129805088043213, 0.30050110816955566, 1.017566442489624, 0.6195608973503113]], [[-0.3941349387168884, 0.7366499900817871, 0.40398189425468445, -0.7567050457000732, 0.8514221906661987, 0.7594303488731384, -0.9627148509025574, 0.9945998191833496]], [[0.9670621752738953, -0.4502611756324768, -0.4575781524181366, 0.059052400290966034, -2.407348871231079, 0.3203718960285187, 0.9106119275093079, 0.1044824868440628]], [[0.3166096806526184, -1.0421128273010254, 0.37535426020622253, 0.33118969202041626, -0.5744143724441528, 0.1520928591489792, -1.2250162363052368, -0.7944929003715515]], [[-0.04402835667133331, -0.8324891328811646, -1.072764277458191, -1.189628005027771, 0.3410857915878296, 1.0265488624572754, 1.2667452096939087, 0.008510381914675236]], [[-0.2054239809513092, -0.016255157068371773, 0.9043656587600708, -1.0108083486557007, -0.0569097101688385, 0.12237723916769028, -0.36187744140625, 1.230217695236206]]], [[[1.5713118314743042, 0.18823587894439697, 1.0011441707611084, -0.8182663321495056, 0.9296925067901611, -1.6218727827072144, -0.31846341490745544, 1.1756529808044434]], [[-0.7054431438446045, 0.5931043028831482, -1.0679279565811157, 0.08812177181243896, 0.12540508806705475, -0.3073638081550598, -0.8744261860847473, -0.3944861888885498]], [[-1.0484989881515503, -0.8986002206802368, -1.1938445568084717, -0.3069351017475128, 1.1766713857650757, 2.4537768363952637, 0.5493519902229309, 0.43762820959091187]], [[-0.8518537282943726, -0.7007747292518616, 0.8302022218704224, 1.1690584421157837, -0.8054897785186768, -0.43780744075775146, 1.218001365661621, 1.2610498666763306]], [[-0.25739893317222595, 1.9919791221618652, -0.9957233667373657, -0.032326724380254745, -0.1812969595193863, -1.0313743352890015, -0.5409165620803833, 0.1828678399324417]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[-0.2529453728676642, 0.01307373129683455], [1.3239921689651746, 0.3622251560101726], [0.2688793121829476, -0.9034700058054191], [-0.24367528488794588, -0.8905112220191892]]], "dtype": "float64"}, {"__nd__": [[[[-0.9127904438019581, 1.1279679587090752], [0.3855097105681205, -1.459021614753764], [0.5267236524677041, 0.7398596965078429], [-0.9791792813951461, -0.48316207788618964]], [[-0.37527355967680454, 0.6883909989698733], [0.7543108460154394, -0.040081700294349286], [-0.061168186571048846, 0.05035019754522335], [-1.711931010226481, -1.764062296081293]]], [[[0.9720123457616624, -0.11124241960127287], [-0.18977936863898515, -1.1426907526001742], [-0.9354900786378098, -1.5001238966603896], [0.8960270338130002, -0.3711260140370331]], [[0.4298398486416093, -0.6755396868472101], [-0.46808235428238004, 1.5414460078233336], [0.5917450943263203, -0.7326938037012146], [-1.0367954389684964, -0.7144030334575311]]], [[[-0.2217525498858756, -1.5480429346289442], [1.8363778816024727, 0.6635083531261197], [0.44014923357034763, -3.4178384256672736], [-2.0479274391782587, 2.0977938521779813]], [[0.6288111801136248, 1.4932527581119812], [1.6738417209050136, 2.2103123185196263], [1.8918981903394547, 0.2839559832181727], [0.22659200367391882, 0.724577403867651]]]], "dtype": "float64"}, {"__nd__": [[[[0.8452550334413007, -0.4037975622073568], [-0.4310657353111702, 0.27786600059276906], [-1.0953748506113832, -2.1175710746264294], [1.6057009435888343, 0.401677978928227]], [[1.6180113290860725, 1.1227334858743905], [1.1532727118752812, 0.927185473187894], [0.800646224746031, 0.9470698239186246], [-0.9427628742784689, -0.47855670307505505]]], [[[1.1954524041866033, -0.45344799035788286], [-1.1295370671926979, 1.5843780953531532], [-2.1345804324247353, 0.4991885588696906], [-0.3555261654274707, 0.45539570049527306]], [[-0.3262707526015803, -0.3862249943525223], [0.042843307181527374, 0.4406757058506384], [-0.9154723630239503, -0.25986876501367345], [-0.4240617097542208, 1.9426691022122367]]], [[[-1.0484101256888392, 1.0415944735899536], [-1.5659903586631365, -1.0034689320212125], [1.2456208954383288, -0.7273694208292941], [1.4936612923008874, 1.8458098551838102]], [[0.3281508180517043, 0.36495256500197343], [-1.5240180263490506, -0.6170714518853393], [0.6246087533300368, -1.8251247316052148], [1.0423100116526938, -0.24861458739618275]]]], "dtype": "float64"}, {"__nd__": [[2, 2, 0]], "dtype": "int64"}, {"__nd__": [6], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[0.19430318474769592, 0.5752548575401306], [-1.3459937572479248, -0.6512214541435242], [1.0946695804595947, -0.8121652603149414], [0.1483231782913208, -0.02717350237071514]]], "dtype": "float32"}, {"__nd__": [[[[-0.22175255417823792, -1.548042893409729], [1.8363778591156006, 0.6635083556175232], [0.4401492476463318, -3.4178383350372314], [-2.0479273796081543, 2.0977938175201416]], [[0.6288111805915833, 1.4932527542114258], [1.6738417148590088, 2.2103123664855957], [1.8918981552124023, 0.28395599126815796], [0.22659200429916382, 0.7245774269104004]], [[-0.22175255417823792, -1.548042893409729], [1.8363778591156006, 0.6635083556175232], [0.4401492476463318, -3.4178383350372314], [-2.0479273796081543, 2.0977938175201416]], [[0.6288111805915833, 1.4932527542114258], [1.6738417148590088, 2.2103123664855957], [1.8918981552124023, 0.28395599126815796], [0.22659200429916382, 0.7245774269104004]], [[-0.9127904176712036, 1.1279679536819458], [0.3855096995830536, -1.4590215682983398], [0.5267236232757568, 0.7398597002029419], [-0.9791792631149292, -0.4831620752811432]], [[-0.37527355551719666, 0.6883909702301025], [0.7543108463287354, -0.0400816984474659], [-0.061168186366558075, 0.05035019665956497], [-1.7119309902191162, -1.7640622854232788]]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[-0.7226732954511434, 3.21911122026557, -4.235699756339064, -1.7475124339693018], [2.4593460344493954, -1.415614848333804, -2.0413287638202173, 2.543478961367531]], [[-0.34959880290373213, -4.697333059865349, -1.0393469516975602, 0.9385147965340201], [1.0227668888308614, 1.9172462386743558, 5.794523697854389, -3.0016910238803356]], [[-0.8298351171234734, 3.1349800704953172, -0.8702795164441722, 0.7745861399865557], [2.4457412876187545, 0.6297817817135174, 1.1283471091393968, 1.504512326994798]], [[1.2958949175154324, -4.812928790377713, 2.001478371190652, 0.10180677960792761], [-2.3255836623545734, -4.165207349086011, 0.32570362668237157, -2.321877643842074]]], "dtype": "float64"}, {"__nd__": [[[[0.4482499667213949, -0.7408374459923583, 0.6749323857284949, -0.8348832543471058], [-0.5993530087252927, -0.6376165151529276, 1.0627860077981546, 0.7105573646048328]], [[0.17439229573373094, -0.41682520560534314, 0.5747922560694333, 1.6378838958678983], [0.006205159282321628, 0.9576433510923413, 0.393389356067412, -0.8546811280481513]]], [[[-1.2081439249946984, -0.2159236938088058, -1.0575328831151432, 0.11143135889331382], [0.47964387731702873, -2.650073235548779, -0.24755150944139062, 1.4380910386947572]], [[-1.3262000818023578, -1.498818370578056, -0.8121748733316944, -0.797108502489324], [0.9847988140433932, -0.5205004540008785, 1.8376411784183702, 0.355796560113378]]], [[[-0.7436870703461929, 0.19649950676276062, 1.590326177861164, -0.15527010047832795], [0.14390246500840895, -1.3563394474528654, -0.5760219636155023, 0.6396818656025394]], [[-0.7589389558299432, 2.6519133165530753, 0.18247579724809132, -1.1408694009774516], [-1.0084698897343134, 0.636808817311084, -0.21459338771386274, -1.2924851330487495]]], [[[1.0048820590016414, -0.2603995596493674, -0.25674091780985425, -0.6404409617654945], [-1.2465742435313303, -1.472932077407791, 0.8648479537594193, 1.3591822060986005]], [[0.6069850272925159, -0.903791598052839, -0.16872603233349817, 1.0245681215497298], [-0.5292506580437172, 0.580898651094816, -1.2023214641458397, 0.5196323644133921]]], [[[0.9748484420476532, 0.8184175913816971, -0.5732103959229595, -0.1916424356441014], [-1.1359874382757238, -0.5512211264933929, 0.37023579919291916, -1.0249777192909295]], [[0.5236324171454144, -1.377762793538944, -0.6762932621075433, 0.9558506603878655], [-1.318393413592343, 0.031231007634365645, 1.0077028956809984, 0.0880507986794551]]], [[[-0.9119431408555221, -0.17937663517782876, 0.5937963330496469, 1.3257583622634592], [-0.4707448204907325, 0.12523369133256695, 0.27758281100750404, -0.9807896688409528]], [[0.4848268710426136, 0.2825866202470884, -1.5211904672820644, -0.1548362031673524], [-0.10098326265005916, -0.6954325838789095, 0.6276614948354511, 0.7088585681907928]]], [[[0.19942969787689624, -0.9279081843555212, -0.9045296073940393, 0.8837852061739353], [-1.9821151719936785, -0.44289912117335567, -0.9655423489299441, -0.2024201490418281]], [[-0.8801235041165545, -0.42846413901169, 0.942830137353368, -1.0177458955972414], [1.4856158961256603, -0.9504074712273219, 0.2048263894994261, -1.0932688896792386]]], [[[1.6345648833118538, -0.3960822857009004, 0.687952222817904, 0.6327877853051946], [0.5534915120029529, 0.5921593544289017, 0.4217043885527952, 0.37326790769010837]], [[0.3540098497925854, -1.0189393639297057, 0.05156554494494281, 0.29923860611876313], [-0.33606165956707906, -0.4576471554421202, -1.1957209895731546, 0.8753358217742514]]], [[[-0.4216742998012845, 1.7667668983354294, 1.6434118644151812, -0.5341263930679222], [-0.2256282304776125, 0.07168833453251083, 2.061033735985658, 0.014798250318352064]], [[0.14685551640005465, 0.6219964618253595, 1.6103208974425285, -0.06195816772275791], [-1.1958405348202579, 0.7783710570843195, 1.286505944090962, 0.6362304570362599]]], [[[1.36446112353404, -0.1789653221206909, -0.11151227165427464, 1.439693949661253], [-1.045282312730405, -1.0958598354679538, 1.3623065018734322, -1.016767262543925]], [[1.274036792264335, -0.6902810779001006, 1.1167389572600146, 2.3161816987802117], [1.3980801203139335, 0.6504531842044562, 0.2795556623222922, 0.7752970129665709]]]], "dtype": "float64"}, {"__nd__": [[[[0.2055484091484459, -0.31929861576639595, 0.9620957859177204, -0.40742264874352013], [0.5219729680380373, -0.6463592388656922, -0.958784042864852, 1.5139964410058215]], [[-0.25514029249782194, 1.0370945479836724, -0.1882064495914043, -0.6104292939294746], [-1.031515805994618, -1.7449655010511733, -0.6301267875561629, -0.6255463564585334]]], [[[1.4364930047636202, -0.6242743264804206, 0.17895455049928047, 0.3504657717341133], [-0.385844027894438, 2.067184355368544, 2.225245508876335, 0.6782362599485624]], [[-1.0298224109350578, -0.939141650091516, 0.5770058739346119, 0.105054374221988], [0.07694407065527276, 0.8398426590692852, 0.039590228231627896, -0.7037044294730789]]], [[[-1.1956108771551068, 0.0785117027369927, -0.45652203791631385, 0.13640428666096774], [-0.7286289292499439, -0.0694010078032111, 0.6724463450064851, -1.4687035004714601]], [[0.5405608147823779, 0.8041346278053069, -1.3343426934467753, -0.9467232023431306], [-0.027462229817727748, -0.6353794343991385, 2.0352491193541202, -1.4344789317200033]]], [[[-0.10581245475837231, -0.3282850575440502, 1.0830616538883655, 0.2041037953189657], [0.179579120194531, 0.035072638818653246, 0.5357927116058496, 0.7884387785804544]], [[-0.8122623804740241, -0.48076243311069206, 0.9843533407503214, -0.5417862520345509], [0.2128692724811487, 0.32695301629101964, -1.5781189240320326, 1.1453462167213053]]], [[[0.00999798289184781, -0.14771184104941057, 1.0406014935343926, -0.4339981294772685], [0.48098809021281436, -0.22597677229973007, -1.1167769804211536, -0.4867072295093602]], [[-0.18226996606425538, -0.06020526575580882, -2.429591317101631, 0.7921823614804946], [-0.5705826504844349, -2.8465219035543265, 1.3570587176317748, -0.3433495571171074]]], [[[-0.1477874561174066, -1.3184668793420664, 0.08066444075788474, 0.43231593678381924], [0.29331496870201396, 1.3892006198082727, 1.5322284029360644, 0.09601515942587377]], [[-0.6483577246148889, -0.6917657016592251, -0.8990004048805413, 1.8373591189939864], [-0.4553464659918596, -0.5057663912460995, 0.16492368848830166, -0.7494856243103536]]], [[[-0.19950129712920084, -0.5480546709020511, 0.4618609903612522, 0.028366716864453242], [-0.3951573743160482, 0.03977879501155993, 0.9526222289340606, -0.29338775009725765]], [[-1.8630868826411782, 0.5814779970215744, -0.33198399112741805, -0.8538085404683052], [0.24014300658011925, -1.0638394394209487, 3.165310714642454, -0.6746890166683034]]], [[[0.9565797456781778, 0.7491966415397067, 0.7097832131508814, 1.116882568736525], [-1.456159033532698, 1.8694635113751497, -0.9826407277618171, -0.52911054551705]], [[-1.1090355933041751, 0.7577921224241303, 0.14150415981347178, -0.4952179444455391], [2.8869907464674793, -0.03510547248361283, 1.4715056292276325, -0.5901543326339712]]], [[[1.4028393812674838, -0.03395916625585613, 0.251809454604433, -1.281065681928143], [2.181017770112795, -2.0132335606288727, -0.376372567907984, -0.3011355418799655]], [[0.20397371402950307, -0.9250253028067392, 1.8228897562135309, -0.145813962285673], [-1.4947802168310198, 1.7990975266999967, -0.8946070303080252, -0.5400525544228212]]], [[[-0.1744475192899677, 0.9913302208800371, 2.1097185174493123, -0.05470314780041465], [-0.8242465189021583, -0.5025442649158055, 1.645806083064104, 1.1484200009390555]], [[-0.26602959210489263, 1.023643643010185, -0.900090106296717, 0.2986143008983277], [0.22811841483456663, 0.9925188746058423, 0.6481943355900096, 0.5617221513346622]]]], "dtype": "float64"}, {"__nd__": [[9, 8, -1, -1], [0, 1, 2, 3], [7, -1, -1, -1], [6, 5, 4, -1]], "dtype": "int64"}, {"__nd__": [3, 8, 1, 6], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[1.1729944944381714, 0.11538664996623993, 0.5155536532402039, -1.1017252206802368], [0.24634285271167755, 0.9467787742614746, 0.6411634087562561, 0.5543613433837891]], [[-0.7499245405197144, -0.6807591319084167, 0.6066779494285583, -0.06311378628015518], [-0.11781992018222809, 0.3039083480834961, -0.04765654355287552, -0.6642685532569885]], [[0.9565797448158264, 0.7491966485977173, 0.7097831964492798, 1.116882562637329], [-1.456158995628357, 1.869463562965393, -0.9826407432556152, -0.5291105508804321]], [[-0.394394189119339, -0.12151151150465012, -1.5896881818771362, 0.4555912911891937], [0.09208499640226364, -0.25494974851608276, 0.09714952111244202, -0.4043045938014984]]], "dtype": "float32"}, {"__nd__": [[[[1.364461064338684, -0.17896531522274017, -0.11151227355003357, 1.4396939277648926], [-1.0452823638916016, -1.0958598852157593, 1.3623064756393433, -1.0167672634124756]], [[1.2740367650985718, -0.690281093120575, 1.1167389154434204, 2.3161816596984863], [1.3980801105499268, 0.6504532098770142, 0.27955564856529236, 0.7752969861030579]], [[-0.42167431116104126, 1.766766905784607, 1.6434118747711182, -0.5341264009475708], [-0.22562822699546814, 0.07168833166360855, 2.0610337257385254, 0.014798250049352646]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]], [[[0.4482499659061432, -0.7408374547958374, 0.6749323606491089, -0.8348832726478577], [-0.599353015422821, -0.6376165151596069, 1.0627859830856323, 0.7105573415756226]], [[0.17439229786396027, -0.41682520508766174, 0.5747922658920288, 1.6378839015960693], [0.006205159239470959, 0.9576433300971985, 0.39338934421539307, -0.854681134223938]], [[-1.208143949508667, -0.21592369675636292, -1.0575329065322876, 0.11143136024475098], [0.47964388132095337, -2.650073289871216, -0.24755151569843292, 1.4380910396575928]], [[-1.3262001276016235, -1.4988183975219727, -0.8121748566627502, -0.79710853099823], [0.984798789024353, -0.5205004811286926, 1.8376411199569702, 0.35579654574394226]], [[-0.743687093257904, 0.1964995115995407, 1.590326189994812, -0.15527009963989258], [0.1439024657011032, -1.356339454650879, -0.5760219693183899, 0.639681875705719]], [[-0.7589389681816101, 2.6519134044647217, 0.18247579038143158, -1.140869379043579], [-1.0084699392318726, 0.6368088126182556, -0.21459338068962097, -1.2924851179122925]], [[1.0048820972442627, -0.26039955019950867, -0.256740927696228, -0.6404409408569336], [-1.2465742826461792, -1.4729321002960205, 0.8648479580879211, 1.3591822385787964]], [[0.6069850325584412, -0.903791606426239, -0.16872602701187134, 1.0245680809020996], [-0.5292506814002991, 0.580898642539978, -1.2023214101791382, 0.5196323394775391]]], [[[1.6345648765563965, -0.3960822820663452, 0.6879522204399109, 0.6327877640724182], [0.5534915328025818, 0.5921593308448792, 0.42170438170433044, 0.37326791882514954]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]], [[[0.1994296908378601, -0.927908182144165, -0.9045296311378479, 0.8837851881980896], [-1.9821151494979858, -0.44289910793304443, -0.9655423760414124, -0.20242014527320862]], [[-0.880123496055603, -0.4284641444683075, 0.9428301453590393, -1.017745852470398], [1.485615849494934, -0.9504074454307556, 0.20482638478279114, -1.093268871307373]], [[-0.9119431376457214, -0.17937663197517395, 0.5937963128089905, 1.3257583379745483], [-0.4707448184490204, 0.1252336949110031, 0.2775828242301941, -0.9807896614074707]], [[0.48482686281204224, 0.28258663415908813, -1.5211905241012573, -0.15483620762825012], [-0.10098326206207275, -0.6954326033592224, 0.6276614665985107, 0.7088585495948792]], [[0.9748484492301941, 0.8184176087379456, -0.5732104182243347, -0.19164243340492249], [-1.135987401008606, -0.5512211322784424, 0.370235800743103, -1.024977684020996]], [[0.5236324071884155, -1.377762794494629, -0.6762932538986206, 0.9558506608009338], [-1.3183934688568115, 0.03123100847005844, 1.0077029466629028, 0.08805079758167267]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(paged_decode, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")